## Data loading

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import optuna
import mlflow
import time
import os

In [2]:
df = pd.read_csv('../data/raw/netflix_customer_churn.csv')
df.head()

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


In [ ]:
df.describe(include='all')

## Visualization

In [ ]:
num_cols = df.select_dtypes(include="number").columns.difference(["churned"])
cat_cols = df.select_dtypes(include="object").columns.difference(["customer_id"])

# inspect correlations of numerical predictor variables
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.tight_layout()
plt.show()

# inspect frequency counts of categorical predictors
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for col, ax in zip(cat_cols, axes.flat):
    df[col].value_counts().plot(kind="bar", ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# dedicated plot for the target variable
plt.figure(figsize=(10, 8))
df["churned"].value_counts().plot(kind="bar")
plt.xticks([0, 1], ["Not Churned", "Churned"], rotation=0)
plt.ylabel("Count")
plt.show()

## One-Hot Encoding

In [ ]:
# get all categorical columns
cat_cols = df.select_dtypes(include="object").columns.difference(["customer_id"])

# apply one-hot encoding to all categorical columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

## Cleaning

In [ ]:
# drop customer ID as it carries no information
df = df.drop('customer_id', axis=1)

# convert booleans to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# inspect presence of NA values
any(df.isna().sum())

## Look for multicolinearity (VIF)

In [ ]:
# quick helper to check multicollinearity
def check_multicollinearity(X):
    vif_data = pd.DataFrame()
    vif_data['feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif_data = vif_data.sort_values(by='VIF', ascending=False)
    print(vif_data)


# run VIF on all features except the target
X = df.drop(columns=['churned'])
check_multicollinearity(X)

# remove monthly-fee column since it is strongly collinear with subscription type
df = df.drop(columns=["monthly_fee"])
X = df.drop(columns=['churned'])
print("-"*100)
check_multicollinearity(X)


## Hyperparameter Tuning (Optuna) via Cross-Validation

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
N_TRIALS = 30
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def _cv_recall(model):
    scores = cross_val_score(model, X_train, y_train, cv=CV, scoring="recall", n_jobs=-1)
    return scores.mean()


def objective_rf(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 600),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        class_weight="balanced",
        random_state=42,
    )
    return _cv_recall(model)


def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 0, 5),
        reg_lambda=trial.suggest_float("reg_lambda", 0, 5),
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
    )
    return _cv_recall(model)


def objective_mlp(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    n_units = trial.suggest_int("n_units", 32, 256)
    mlp = MLPClassifier(
        hidden_layer_sizes=tuple([n_units] * n_layers),
        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
        alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        batch_size=trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        max_iter=300,
        early_stopping=True,
        random_state=42,
    )
    pipeline = Pipeline([("scaler", StandardScaler()), ("mlp", mlp)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=CV, scoring="recall", n_jobs=-1)
    return scores.mean()


print("Tuning RandomForest...")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=N_TRIALS)
print(f"  Best CV Recall: {study_rf.best_value:.4f}")

print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
print(f"  Best CV Recall: {study_xgb.best_value:.4f}")

print("Tuning MLP...")
study_mlp = optuna.create_study(direction="maximize")
study_mlp.optimize(objective_mlp, n_trials=N_TRIALS)
print(f"  Best CV Recall: {study_mlp.best_value:.4f}")

## Evaluate on the Test Set

In [ ]:
# set a lower threshold to enable better recall
THRESHOLD = 0.4

# helper to build the MLP
def build_mlp(best_params):
    n_layers = best_params.pop("n_layers")
    n_units = best_params.pop("n_units")
    mlp = MLPClassifier(
        hidden_layer_sizes=tuple([n_units] * n_layers),
        max_iter=300,
        early_stopping=True,
        random_state=42,
        **best_params,
    )
    return Pipeline([("scaler", StandardScaler()), ("mlp", mlp)])

# store results and trained models
results = []
trained_models = {}

configs = [
    ("RandomForest", RandomForestClassifier, study_rf.best_params,
     {"class_weight": "balanced", "random_state": 42, "n_jobs": -1}),
    ("XGBoost", XGBClassifier, study_xgb.best_params,
     {"scale_pos_weight": scale_pos_weight, "random_state": 42, "n_jobs": -1, "eval_metric": "logloss"}),
]

# loop over trees and evaluate on the test set
for name, ModelClass, best_params, fixed_params in configs:
    params = {**best_params, **fixed_params}
    model = ModelClass(**params)
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int)
    results.append({
        "Model": name,
        "ROC-AUC": round(roc_auc_score(y_test, proba), 4),
        "F1 (churn)": round(f1_score(y_test, y_pred, pos_label=1), 4),
        "Recall (churn)": round(recall_score(y_test, y_pred, pos_label=1), 4),
        "Precision (churn)": round(precision_score(y_test, y_pred, pos_label=1), 4),
        "Train time (s)": round(train_time, 2),
    })
    trained_models[name] = model

# evaluate MLP on the test set
mlp_params = dict(study_mlp.best_params)
mlp_pipeline = build_mlp(mlp_params)
t0 = time.time()
mlp_pipeline.fit(X_train, y_train)
mlp_train_time = time.time() - t0
mlp_proba = mlp_pipeline.predict_proba(X_test)[:, 1]
mlp_pred = (mlp_proba >= THRESHOLD).astype(int)
results.append({
    "Model": "MLP",
    "ROC-AUC": round(roc_auc_score(y_test, mlp_proba), 4),
    "F1 (churn)": round(f1_score(y_test, mlp_pred, pos_label=1), 4),
    "Recall (churn)": round(recall_score(y_test, mlp_pred, pos_label=1), 4),
    "Precision (churn)": round(precision_score(y_test, mlp_pred, pos_label=1), 4),
    "Train time (s)": round(mlp_train_time, 2),
})
trained_models["MLP"] = mlp_pipeline

# show the comparison in a pandas df
summary = pd.DataFrame(results).set_index("Model").sort_values("Recall (churn)", ascending=False)
display(summary.head())

## Log Comparison to MLlow

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
mlflow.set_tracking_uri(f"file://{project_root}/mlruns")
mlflow.set_experiment("Churn - Architecture Comparison")

mlflow_params = {
    "RandomForest": study_rf.best_params,
    "XGBoost": study_xgb.best_params,
    "MLP": study_mlp.best_params,
}

for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int)

    with mlflow.start_run(run_name=name):
        mlflow.log_params(mlflow_params[name])
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, proba))
        mlflow.log_metric("f1_churn", f1_score(y_test, y_pred, pos_label=1))
        mlflow.log_metric("recall_churn", recall_score(y_test, y_pred, pos_label=1))
        mlflow.log_metric("precision_churn", precision_score(y_test, y_pred, pos_label=1))
        mlflow.sklearn.log_model(model, "model")

print("All runs logged to MLflow.")